In [ ]:
# Copyright Aditya Rane
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Session 3: Window Functions + Top Interview Patterns
> This notebook introduces Session 3: Window Functions + Top Interview Patterns

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/Adi8885/python-for-beginners/blob/main/tutorial-May2026-InterviewPrep-Batch/Session1%20-%20SQL%20Mastery%20—%20The%20Foundations.ipynb">
      <img src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Run in Colab
    </a>
  </td>
    <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https://raw.githubusercontent.com/Adi8885/python-for-beginners/refs/heads/main/tutorial-May2026-InterviewPrep-Batch/Session1%20-%20SQL%20Mastery%20—%20The%20Foundations.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/Adi8885/python-for-beginners/blob/main/tutorial-May2026-InterviewPrep-Batch/Session1%20-%20SQL%20Mastery%20—%20The%20Foundations.ipynb">
      <img src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/Adi8885/python-for-beginners/refs/heads/main/tutorial-May2026-InterviewPrep-Batch/Session1%20-%20SQL%20Mastery%20—%20The%20Foundations.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
</table>

# 🎓 Session 3: Window Functions & Advanced SQL Patterns

| Item | Details |
| :--- | :--- |
| ⏳ **Duration** | 90–120 minutes |
| 🎯 **Level** | Intermediate |
| 🚀 **Main Goal** | Students should be able to solve ranking, latest record, running total, and comparison-based SQL problems. |

---

## 🌟 Why This Session?
In SQL interviews and data analytics roles, **Window Functions** are the most highly-tested topic! Why? Because they allow you to perform advanced calculations (like running totals, moving averages, and ranks) while keeping the original rows intact. They turn complex self-joins into single, elegant lines of SQL. 💼✨

---

## 🧩 Part 1: Why Window Functions Are Needed — 10 mins

Let's understand the difference between `GROUP BY` and window functions.

### 1. `GROUP BY` (The Collapser 🧹)
`GROUP BY` collapses multiple rows into a single summary row. Once grouped, you lose the individual row details.

**Query:**
```sql
SELECT customer_id, SUM(amount) AS total_amount
FROM orders
GROUP BY customer_id;
```

**Visual Output:**
| customer_id | total_amount |
| :--- | :--- |
| `A` | `30` |
| `B` | `15` |

---

### 2. Window Function (The Calculator 🧮)
A **Window Function** performs calculations across a set of rows (a "window") related to the current row, but **retains all individual rows** and appends the result as a new column.

**Query:**
```sql
SELECT
  customer_id,
  order_id,
  amount,
  SUM(amount) OVER(PARTITION BY customer_id) AS customer_total
FROM orders;
```

**Visual Output:**
| customer_id | order_id | amount | customer_total |
| :--- | :--- | :--- | :--- |
| `A` | `1` | `10` | **`30`** |
| `A` | `2` | `20` | **`30`** |
| `B` | `3` | `15` | **`15`** |

### 📊 Visualizing the Difference

```mermaid
graph TD
    subgraph GroupBy [GROUP BY - Collapses Rows]
        A1["Row 1: Customer A ($10)"] --> G1["Group Customer A"]
        A2["Row 2: Customer A ($20)"] --> G1
        A3["Row 3: Customer B ($15)"] --> G2["Group Customer B"]
        G1 --> Out1["Output 1: Customer A, $30"]
        G2 --> Out2["Output 2: Customer B, $15"]
    end
    
    subgraph WindowFunc [Window Function - Retains Rows]
        W1["Row 1: Customer A ($10)"] --> Win1["Output 1: Customer A, $10, Total: $30"]
        W2["Row 2: Customer A ($20)"] --> Win2["Output 2: Customer A, $20, Total: $30"]
        W3["Row 3: Customer B ($15)"] --> Win3["Output 3: Customer B, $15, Total: $15"]
    end
```

---

## 🏗️ Part 2: The Core Anatomy of a Window Function — 15 mins

To invoke a window function, we use the `OVER()` clause. The basic structure is:

```sql
FUNCTION() OVER (
    PARTITION BY partition_column
    ORDER BY sort_column
    ROWS/RANGE frame_clause
)
```

- **`PARTITION BY`** 🍰: Slices your data into groups (or partitions). The window calculation restarts for each group.
- **`ORDER BY`** 📶: Sorts the rows *within* each partition. Essential for ranking and running totals.
- **`ROWS/RANGE`** 🖼️: Limits the rows within the partition used for the calculation (e.g. "preceding 3 rows").

---

## 👑 Part 3: Ranking Problems (`ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`) — 25 mins

Understanding the differences between these ranking functions is a classic interview question.

### 📝 The Ranking Comparison Table

| Function | Behavior | Tie Handling | Example Sequence (1st, 1st, 3rd) |
| :--- | :--- | :--- | :--- |
| **`ROW_NUMBER()`** | Sequential integers starting at 1. | Assigns arbitrary order for ties. | `1, 2, 3` |
| **`RANK()`** | Rank with gaps after ties. | Tied rows get the same rank; next rank is skipped. | `1, 1, 3` |
| **`DENSE_RANK()`** | Rank without gaps after ties. | Tied rows get the same rank; next rank is sequential. | `1, 1, 2` |

### 🎬 IMDb Example: Ranking Marathi Movies by Rating
Let's run a query to rank Marathi movies.

```sql
SELECT l.Name AS Language,
       m.title AS Movie_Title,
       m.rating AS Rating,
       ROW_NUMBER() OVER(PARTITION BY l.Name ORDER BY m.rating DESC) AS Row_Num,
       RANK() OVER(PARTITION BY l.Name ORDER BY m.rating DESC) AS Rank_Num,
       DENSE_RANK() OVER(PARTITION BY l.Name ORDER BY m.rating DESC) AS Dense_Rank_Num
FROM Movie m
INNER JOIN M_Language ml ON m.MID = ml.MID
INNER JOIN Language l ON ml.LAID = l.LAID
WHERE l.Name = 'Marathi'
ORDER BY m.rating DESC;
```

**Results:**
| Language | Movie_Title | Rating | Row_Num | Rank_Num | Dense_Rank_Num |
| :--- | :--- | :--- | :--- | :--- | :--- |
| Marathi | Tumbbad | 8.5 | 1 | 1 | 1 |
| Marathi | Ek Hota Vidushak | 7.9 | 2 | 2 | 2 |
| Marathi | Kaksparsh | 7.8 | 3 | 3 | 3 |
| Marathi | Kairee | 7.7 | 4 | **4** | **4** |
| Marathi | Court | 7.7 | 5 | **4** | **4** |
| Marathi | Pinjra | 7.6 | 6 | **6** *(skips 5)* | **5** *(dense)* |

---

## 🎯 Part 4: Latest Record Pattern (Filtering Rankings) — 20 mins

**Goal:** Find the highest-rated movie for each language.

### ⚠️ The Execution Order Trap!
Why can't you do this?
```sql
-- ❌ ERROR!
SELECT title, ROW_NUMBER() OVER(PARTITION BY LAID ORDER BY rating DESC) AS rn
FROM Movie
WHERE rn = 1;
```
Because SQL's logical execution order runs the `WHERE` clause **before** the `SELECT` clause (where window functions are calculated). Therefore, `rn` does not exist when `WHERE` runs.

### 🔑 The Fix: Use a CTE (Common Table Expression)
```sql
WITH RankedMovies AS (
    SELECT l.Name AS Language_Name,
           m.title AS Movie_Title,
           m.rating AS Rating,
           ROW_NUMBER() OVER(PARTITION BY l.Name ORDER BY m.rating DESC) AS rn
    FROM Movie m
    INNER JOIN M_Language ml ON m.MID = ml.MID
    INNER JOIN Language l ON ml.LAID = l.LAID
)
SELECT Language_Name, Movie_Title, Rating
FROM RankedMovies
WHERE rn = 1
ORDER BY Rating DESC;
```

---

## 📈 Part 5: Running Totals (Cumulative Sums) — 20 mins

A running total is a cumulative sum of data. It is calculated by using `SUM()` with `ORDER BY` inside `OVER()`.

### 🎬 IMDb Example: Running Total of Movies Released (2010–2018)
```sql
SELECT year AS Release_Year,
       COUNT(*) AS Movies_Released,
       SUM(COUNT(*)) OVER(ORDER BY year) AS Cumulative_Movies
FROM Movie
WHERE year >= '2010' AND year <= '2018'
GROUP BY year
ORDER BY year;
```

**Results:**
| Release_Year | Movies_Released | Cumulative_Movies |
| :--- | :--- | :--- |
| `2010` | `117` | `117` |
| `2011` | `109` | `226` *(117 + 109)* |
| `2012` | `108` | `334` *(226 + 108)* |
| `2013` | `127` | `461` |
| `2014` | `118` | `579` |
| `2015` | `109` | `688` |
| `2016` | `118` | `806` |
| `2017` | `118` | `924` |
| `2018` | `93` | `1017` |

---

## 🔄 Part 6: Comparison-Based Problems (`LAG()` and `LEAD()`) — 25 mins

Sometimes you need to compare a value in the current row to a value in a previous or subsequent row.
- **`LAG(column, offset)`**: Looks backward (returns value from `offset` rows before).
- **`LEAD(column, offset)`**: Looks forward (returns value from `offset` rows after).

### 🎬 IMDb Example: YoY Change in USA Movies
```sql
WITH USA_Annual_Movies AS (
    SELECT year AS Release_Year,
           COUNT(*) AS Movie_Count
    FROM Movie m
    INNER JOIN M_Country mc ON m.MID = mc.MID
    INNER JOIN Country c ON mc.CID = c.CID
    WHERE c.Name = 'USA' AND year >= '2010' AND year <= '2018'
    GROUP BY year
)
SELECT Release_Year,
       Movie_Count,
       LAG(Movie_Count, 1) OVER(ORDER BY Release_Year) AS Prev_Year_Count,
       Movie_Count - LAG(Movie_Count, 1) OVER(ORDER BY Release_Year) AS YoY_Change
FROM USA_Annual_Movies
ORDER BY Release_Year;
```

**Results:**
| Release_Year | Movie_Count | Prev_Year_Count | YoY_Change |
| :--- | :--- | :--- | :--- |
| `2010` | `2` | `NULL` | `NULL` |
| `2011` | `2` | `2` | `0` |
| `2012` | `4` | `2` | `+2` |
| `2013` | `2` | `4` | `-2` |
| `2014` | `6` | `2` | `+4` |
| `2015` | `1` | `6` | `-5` |
| `2016` | `3` | `1` | `+2` |
| `2017` | `1` | `3` | `-2` |
| `2018` | `1` | `1` | `0` |

---

## 🏠 Session 3 Homework: The Window Function Challenge

Test your skills using the `Db-IMDB-SQL-Dataset.db`!

### 📝 The Challenge Set

| No. | Problem Statement | Difficulty | Hint 💡 |
| :--- | :--- | :--- | :--- |
| **1** | For the year `'2018'`, rank all movies using `DENSE_RANK()` based on their rating. Show the highest-rated movies first. | ⭐⭐ | `DENSE_RANK() OVER(ORDER BY rating DESC)` |
| **2** | Find the highest-rated movie for each year. If there is a tie, display the movie with more votes. Use a CTE. | ⭐⭐⭐ | CTE + `ROW_NUMBER() OVER(PARTITION BY year ORDER BY rating DESC, num_votes DESC)` |
| **3** | Calculate the running total of movie votes per year for movies released from `'2015'` to `'2018'`. | ⭐⭐⭐ | Group by year first, then do `SUM(Year_Votes) OVER(ORDER BY year)` |
| **4** | For movies directed by `'Rajkumar Hirani'`, list the title, release year, rating, and the rating of his previous movie (by year) using `LAG()`. Also calculate the difference in rating from his previous movie. | ⭐⭐⭐⭐ | Join `Movie` ↔ `M_Director` ↔ `Person` and use `LAG(rating) OVER(ORDER BY year)` |

**Happy Querying!** 🚀💻

---




# 🎓 Part 4: LAG, LEAD, Running Totals & Moving Averages — 30 mins

| Concept | Purpose | Real-World Analogy 💡 |
| :--- | :--- | :--- |
| **`LAG()`** | Look at the previous row's value. | Checking yesterday's weather forecast. |
| **`LEAD()`** | Look at the next row's value. | Peeking at tomorrow's schedule. |
| **Running Total** | Calculate cumulative sum over time. | Watching a scoreboard accumulate points. |
| **Moving Average** | Smooth out noise by averaging values over a rolling window. | Looking at a 7-day trend line of active users. |

---

## 🔍 Understanding the Core Concepts

### 1. `LAG()` and `LEAD()` (The Time Travelers 🕰️)
These functions allow you to reference rows offset from the current row without doing complex self-joins.

## LAG Function — Looks Backward

| Row | Date & Current Value | LAG(1) Output (Prev Value) | Explanation |
| --- | --- | --- | --- |
| **Row 1** | May 01 ($100) | **NULL** | No previous row available |
| **Row 2** | May 02 ($150) | **$100** | Pulls value from Row 1 |
| **Row 3** | May 03 ($120) | **$150** | Pulls value from Row 2 |

---

> **How it works:** The LAG(1) function shifts the dataset down by 1 row, allowing you to look back at the previous row's value relative to your current row.

*   **Syntax:** `LAG(column, offset, default_value) OVER (ORDER BY column)`
*   **Default offset** is `1` if not specified.
*   **Default value** is `NULL` if there is no previous row.

---

### 2. Running Totals (Cumulative Sums 📈)
By combining `SUM()` with `ORDER BY` in the `OVER()` clause, SQL performs a running sum from the first record up to the current row.

*   **Syntax:** `SUM(column) OVER (ORDER BY date_column)`

---

### 3. Moving Averages (Rolling Windows 🌊)
Useful for removing short-term fluctuations and highlighting longer-term trends.

## Rolling 3-Day Window (Moving Average)

### Window 1: Days 1–3

* **Day 1:** $100
* **Day 2:** $150
* **Day 3:** $200
* **Calculation:** $\frac{100 + 150 + 200}{3} = \mathbf{\$150.00}$

---

### Window 2: Days 2–4

* **Day 2:** $150
* **Day 3:** $200
* **Day 4:** $250
* **Calculation:** $\frac{150 + 200 + 250}{3} = \mathbf{\$200.00}$

---

### Alternative: Comparative Table View

| Timeline | Current Value | Window 1 (Days 1-3) | Window 2 (Days 2-4) |
| --- | --- | --- | --- |
| **Day 1** | $100 | Included |  |
| **Day 2** | $150 | Included | Included |
| **Day 3** | $200 | Included | Included |
| **Day 4** | $250 |  | Included |
| **Result** | **Moving Average:** | **$150.00** | **$200.00** |

*   **Syntax:** 
    ```sql
    AVG(column) OVER (
        ORDER BY date_column
        ROWS BETWEEN N PRECEDING AND CURRENT ROW
    )
    ```

---

## 🏋️ Practice Set 2: Time-Based Analysis

### 📋 Setup Schema
Assume a table named `daily_sales` tracks daily revenue:

| Column | Type | Description |
| :--- | :--- | :--- |
| `sale_date` | `DATE` | The date of sale (Primary Key) |
| `revenue` | `DECIMAL` | The total revenue generated on that day |

---

### ❓ Questions & Solutions

#### **Q1. Calculate previous day's revenue.**
*   **Goal:** Retrieve the revenue of the day before the current row.
*   **Query:**
    ```sql
    SELECT
      sale_date,
      revenue,
      LAG(revenue, 1) OVER (ORDER BY sale_date) AS prev_day_revenue
    FROM daily_sales;
    ```

#### **Q2. Calculate day-over-day revenue change.**
*   **Goal:** Find the absolute difference in revenue between today and yesterday.
*   **Query:**
    ```sql
    SELECT
      sale_date,
      revenue,
      LAG(revenue, 1) OVER (ORDER BY sale_date) AS prev_day_revenue,
      revenue - LAG(revenue, 1) OVER (ORDER BY sale_date) AS dod_revenue_change
    FROM daily_sales;
    ```

#### **Q3. Calculate day-over-day percentage change.**
*   **Goal:** Find the relative change in revenue as a percentage.
*   **Query:**
    ```sql
    SELECT
      sale_date,
      revenue,
      ROUND(
        (revenue - LAG(revenue, 1) OVER (ORDER BY sale_date)) * 100.0 / 
        LAG(revenue, 1) OVER (ORDER BY sale_date), 
        2
      ) AS dod_pct_change
    FROM daily_sales;
    ```
    > [!TIP]
    > In production, you should handle division-by-zero or `NULL` states using a `CASE WHEN` or `NULLIF()` statement if `LAG(...)` can be `0`.

#### **Q4. Calculate running total revenue by date.**
*   **Goal:** Show cumulative revenue built up day by day.
*   **Query:**
    ```sql
    SELECT
      sale_date,
      revenue,
      SUM(revenue) OVER (ORDER BY sale_date) AS running_total_revenue
    FROM daily_sales;
    ```

#### **Q5. Calculate 7-day moving average revenue.**
*   **Goal:** Average the revenue of the current day and the previous 6 days.
*   **Query:**
    ```sql
    SELECT
      sale_date,
      revenue,
      ROUND(
        AVG(revenue) OVER (
          ORDER BY sale_date 
          ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 
        2
      ) AS moving_avg_7day
    FROM daily_sales;
    ```

---

### 🎬 IMDb Live Dataset Challenge: Time-Based Analysis
Let's apply these concepts directly to the `Db-IMDB-SQL-Dataset.db`database! 

Since the `Movie` table has years in mixed formats (e.g., `'XVII 2016'`, `'2018'`), we clean it by extracting the last 4 characters using `SUBSTR(year, -4)` and casting it to an integer.

#### **Exercise A: YoY Percentage Change in Movie Releases (2010–2018)**
```sql
WITH AnnualMovies AS (
    SELECT CAST(SUBSTR(year, -4) AS INTEGER) AS Release_Year,
           COUNT(*) AS Movie_Count
    FROM Movie
    GROUP BY Release_Year
)
SELECT Release_Year,
       Movie_Count,
       LAG(Movie_Count) OVER (ORDER BY Release_Year) AS Prev_Year_Count,
       ROUND(
         (Movie_Count - LAG(Movie_Count) OVER (ORDER BY Release_Year)) * 100.0 / 
         LAG(Movie_Count) OVER (ORDER BY Release_Year), 
         2
       ) AS YoY_Pct_Change
FROM AnnualMovies
WHERE Release_Year BETWEEN 2010 AND 2018
ORDER BY Release_Year;
```

**Results Output:**
| Release_Year | Movie_Count | Prev_Year_Count | YoY_Pct_Change |
| :--- | :--- | :--- | :--- |
| `2010` | `125` | `NULL` | `NULL` |
| `2011` | `116` | `125.0` | `-7.20%` |
| `2012` | `110` | `116.0` | `-5.17%` |
| `2013` | `136` | `110.0` | `+23.64%` |
| `2014` | `126` | `136.0` | `-7.35%` |
| `2015` | `119` | `126.0` | `-5.56%` |
| `2016` | `129` | `119.0` | `+8.40%` |
| `2017` | `126` | `129.0` | `-2.33%` |
| `2018` | `104` | `126.0` | `-17.46%` |

---

#### **Exercise B: 3-Year Moving Average Rating of Movies Over Time (2010–2018)**
```sql
WITH AnnualAvgRating AS (
    SELECT CAST(SUBSTR(year, -4) AS INTEGER) AS Release_Year,
           ROUND(AVG(rating), 2) AS Avg_Rating
    FROM Movie
    GROUP BY Release_Year
)
SELECT Release_Year,
       Avg_Rating,
       ROUND(
         AVG(Avg_Rating) OVER (
           ORDER BY Release_Year
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
         ), 
         2
       ) AS Moving_Avg_3Year
FROM AnnualAvgRating
WHERE Release_Year BETWEEN 2010 AND 2018
ORDER BY Release_Year;
```

**Results Output:**
| Release_Year | Avg_Rating | Moving_Avg_3Year |
| :--- | :--- | :--- |
| `2010` | `5.72` | `5.72` |
| `2011` | `5.63` | `5.67` |
| `2012` | `5.81` | `5.72` |
| `2013` | `5.71` | `5.72` |
| `2014` | `5.71` | `5.74` |
| `2015` | `5.90` | `5.77` |
| `2016` | `5.93` | `5.85` |
| `2017` | `6.17` | `6.00` |
| `2018` | `6.33` | `6.14` |

---

## 🏋️ Practice Set 3: Customer Purchase Analysis

### 📋 Setup Schema
Assume a table named `orders` tracks customer transaction histories:

| Column | Type | Description |
| :--- | :--- | :--- |
| `order_id` | `INT` | Unique ID of the order (Primary Key) |
| `customer_id` | `INT` | ID of the customer who placed the order |
| `order_date` | `DATE` | Date when the order was placed |
| `amount` | `DECIMAL` | Transaction amount paid |

---

### ❓ Questions & Solutions

#### **Q1. Find each customer's first order date.**
*   **Goal:** Identify the earliest transaction date for each unique customer.
*   **Query (Window Function Pattern):**
    ```sql
    WITH RankedOrders AS (
      SELECT
        customer_id,
        order_date,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date ASC) AS rn
      FROM orders
    )
    SELECT customer_id, order_date AS first_order_date
    FROM RankedOrders
    WHERE rn = 1;
    ```
    *(Note: While `MIN(order_date)` with `GROUP BY` is faster here, using CTE + `ROW_NUMBER()` is the standard pattern when you need to fetch the entire order row details, like `amount` or `order_id` of the first order!)*

#### **Q2. Find each customer's latest order date.**
*   **Goal:** Identify the most recent transaction date for each unique customer.
*   **Query:**
    ```sql
    WITH RankedOrders AS (
      SELECT
        customer_id,
        order_date,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS rn
      FROM orders
    )
    SELECT customer_id, order_date AS latest_order_date
    FROM RankedOrders
    WHERE rn = 1;
    ```

#### **Q3. Find the time gap between two consecutive orders.**
*   **Goal:** Calculate the number of days that passed between a customer's order and their preceding order.
*   **Query (Standard SQL):**
    ```sql
    SELECT
      customer_id,
      order_id,
      order_date,
      LAG(order_date, 1) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_order_date,
      order_date - LAG(order_date, 1) OVER (PARTITION BY customer_id ORDER BY order_date) AS days_since_prev_order
    FROM orders;
    ```
    > [!IMPORTANT]
    > **Database Specifics:** Date arithmetic syntax varies. For **SQLite**, replace `order_date - LAG(...)` with:
    > `JULIANDAY(order_date) - JULIANDAY(LAG(order_date, 1) OVER (PARTITION BY customer_id ORDER BY order_date))`
    > For **PostgreSQL**, it is simply subtraction. For **MySQL**, use `DATEDIFF(order_date, LAG(...))`.

#### **Q4. Find customers whose second order happened within 30 days of first order.**
*   **Goal:** Locate users who made a repeat purchase quickly (high-retention cohorts).
*   **Query (Standard SQL):**
    ```sql
    WITH OrderedGaps AS (
      SELECT
        customer_id,
        order_date,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS rn,
        LAG(order_date, 1) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_order_date
      FROM orders
    )
    SELECT DISTINCT customer_id
    FROM OrderedGaps
    WHERE rn = 2 AND (order_date - prev_order_date) <= 30;
    ```
    > [!NOTE]
    > **SQLite syntax version:** `WHERE rn = 2 AND (JULIANDAY(order_date) - JULIANDAY(prev_order_date)) <= 30`

#### **Q5. Calculate cumulative spending for each customer over time.**
*   **Goal:** Sum order amounts chronologically, grouped by customer.
*   **Query:**
    ```sql
    SELECT
      customer_id,
      order_date,
      amount,
      SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS cumulative_spending
    FROM orders;
    ```

---

### 🎬 IMDb Live Dataset Challenge: Director Filmography Analysis
Let's apply these transaction-style patterns to actor/director timelines in the IMDb dataset!

#### **Exercise A: Calculate the Year Gap Between Consecutive Movies Directed by 'Rajkumar Hirani'**
Here, we track how many years passed between each film release directed by Rajkumar Hirani.
```sql
SELECT m.title AS Movie_Title,
       CAST(SUBSTR(m.year, -4) AS INTEGER) AS Release_Year,
       LAG(CAST(SUBSTR(m.year, -4) AS INTEGER)) OVER (ORDER BY CAST(SUBSTR(m.year, -4) AS INTEGER)) AS Prev_Movie_Year,
       CAST(SUBSTR(m.year, -4) AS INTEGER) - 
       LAG(CAST(SUBSTR(m.year, -4) AS INTEGER)) OVER (ORDER BY CAST(SUBSTR(m.year, -4) AS INTEGER)) AS Year_Gap
FROM Movie m
INNER JOIN M_Director md ON m.MID = md.MID
INNER JOIN Person p ON md.PID = p.PID
WHERE TRIM(p.Name) = 'Rajkumar Hirani'
ORDER BY Release_Year;
```

**Results Output:**
| Movie_Title | Release_Year | Prev_Movie_Year | Year_Gap |
| :--- | :--- | :--- | :--- |
| `Munna Bhai M.B.B.S.` | `2003` | `NULL` | `NULL` |
| `Lage Raho Munna Bhai` | `2006` | `2003` | `3` |
| `3 Idiots` | `2009` | `2006` | `3` |
| `PK` | `2014` | `2009` | `5` |
| `Sanju` | `2018` | `2014` | `4` |

---

#### **Exercise B: Find Directors Whose Second Directed Movie Was Within 3 Years of Their First Movie**
This query identifies directors who released their second project shortly after their debut.
```sql
WITH DirectorMovies AS (
    SELECT TRIM(p.Name) AS Director_Name,
           m.title AS Movie_Title,
           CAST(SUBSTR(m.year, -4) AS INTEGER) AS Release_Year,
           ROW_NUMBER() OVER (PARTITION BY p.PID ORDER BY CAST(SUBSTR(m.year, -4) AS INTEGER)) AS rn,
           LAG(CAST(SUBSTR(m.year, -4) AS INTEGER)) OVER (PARTITION BY p.PID ORDER BY CAST(SUBSTR(m.year, -4) AS INTEGER)) AS prev_year
    FROM Movie m
    INNER JOIN M_Director md ON m.MID = md.MID
    INNER JOIN Person p ON md.PID = p.PID
)
SELECT Director_Name, Release_Year, prev_year, (Release_Year - prev_year) AS Year_Gap
FROM DirectorMovies
WHERE rn = 2 AND (Release_Year - prev_year) <= 3 AND prev_year IS NOT NULL
ORDER BY Year_Gap ASC, Director_Name ASC
LIMIT 5;
```

**Results Output:**
| Director_Name | Release_Year | prev_year | Year_Gap (Years) |
| :--- | :--- | :--- | :--- |
| `Abhinay Deo` | `2011` | `2011` | `0` |
| `Akarsh Khurana` | `2018` | `2018` | `0` |
| `Anurag Basu` | `2003` | `2003` | `0` |
| `Bimal Roy` | `1953` | `1953` | `0` |
| `Deepak Anand` | `1992` | `1992` | `0` |

---

#### **Exercise C: Cumulative Movie Ratings Achieved by 'Rajkumar Hirani'**
This query lists the running total rating score achieved by his projects sequentially.
```sql
SELECT m.title AS Movie_Title,
       CAST(SUBSTR(m.year, -4) AS INTEGER) AS Release_Year,
       m.rating AS Movie_Rating,
       SUM(m.rating) OVER (ORDER BY CAST(SUBSTR(m.year, -4) AS INTEGER)) AS Cumulative_Rating
FROM Movie m
INNER JOIN M_Director md ON m.MID = md.MID
INNER JOIN Person p ON md.PID = p.PID
WHERE TRIM(p.Name) = 'Rajkumar Hirani'
ORDER BY Release_Year;
```

**Results Output:**
| Movie_Title | Release_Year | Movie_Rating | Cumulative_Rating |
| :--- | :--- | :--- | :--- |
| `Munna Bhai M.B.B.S.` | `2003` | `8.2` | `8.2` |
| `Lage Raho Munna Bhai` | `2006` | `8.1` | `16.3` *(8.2 + 8.1)* |
| `3 Idiots` | `2009` | `8.4` | `24.7` *(16.3 + 8.4)* |
| `PK` | `2014` | `8.2` | `32.9` |
| `Sanju` | `2018` | `8.1` | `41.0` |

---

## 📚 Homework – Window Functions, LEAD/LAG & Time‑Series (IMDb Dataset)

| #️⃣ | 📝 Question |
|-----|------------|
| 1️⃣ | For each **language**, list the movies ranked by rating (highest first) and include a column showing the **row number** within that language. |
| 2️⃣ | Find the **most‑rated** movie for every **genre**. Return the genre name, movie title, and rating. |
| 3️⃣ | Compute a **running total** of the number of movies released per year from **2010 to 2018**. Show the year, yearly count, and cumulative count. |
| 4️⃣ | Using `LAG`, calculate the **year‑over‑year percentage change** in the count of movies released (for all years). Show year, count, previous year count, and % change. |
| 5️⃣ | For each **director**, show the **rating difference** between a movie and the previous movie they directed (ordered by release year). Include director name, movie title, year, rating, previous rating, and rating delta. |
| 6️⃣ | Determine the **gap in years** between consecutive movies for every director. Return director name, current year, previous year, and the year gap. |
| 7️⃣ | For each **actor**, list the number of movies they appeared in each year and the **difference** from the previous year using `LAG`. Show actor name, year, movies_this_year, movies_previous_year, and diff. |
| 8️⃣ | Calculate a **3‑year moving average** of the average movie rating (by year). Show year, average rating, and the moving‑average value. |
| 9️⃣ | For each **country**, compute the **cumulative sum** of `num_votes` over the years and rank the years by this cumulative total. Show country, year, annual votes, cumulative votes, and rank. |
| 🔟 | Using `LEAD`, find for each **movie** the rating of the **next movie** released in the same language (ordered by year). Show language, current movie title, year, rating, next movie title, next rating, and rating difference. |
| 1️⃣1️⃣ | Identify directors whose **second movie** has a **higher rating** than their first. Return director name, titles of the first and second movies, their ratings, and the rating improvement. |
| 1️⃣2️⃣ | For each **genre**, list movies whose rating **improved** compared to the previous movie of the same genre (ordered by year). Include genre, movie title, year, rating, previous rating, and rating gain. |
| 1️⃣3️⃣ | Compute the **year‑over‑year change** in total `num_votes` for each **country** (using `LAG`). Show country, year, votes, previous year votes, absolute change, and % change. |
| 1️⃣4️⃣ | For each **language**, calculate a **5‑year moving average** of the yearly movie count. Show language, year, yearly count, and the moving‑average. |
| 1️⃣5️⃣ | Determine the **longest streak** of consecutive years in which each actor appeared in at least one movie. Return actor name, start year, end year, and streak length (only include streaks ≥ 3 years). |